# Onderzoeksmethode: Hoe vergelijken, onderbouwen en testen we agenttools?

> **Doel:** een reproduceerbare, schriftelijk onderbouwde aanpak om de juiste agent-framework + LLM + tool-stack te kiezen voor de **Boodschappen Agent** (Node 1–9).

---

## 1. Onderzoeksstrategie in 5 fasen

```
[1] Scoping  →  [2] Desk research  →  [3] Criteria & rubric  →  [4] PoC-test  →  [5] Onderbouwing & keuze
```

| Fase | Doel | Output |
|---|---|---|
| **1. Scoping** | Vaststellen welke categorieën tools relevant zijn voor onze 9-node flow | Long-list per categorie |
| **2. Desk research** | Documentatie, benchmarks, GitHub-activiteit, community-signalen | Short-list (max. 3 per categorie) |
| **3. Criteria & rubric** | Meetbare vergelijkingscriteria met gewichten | Scorekaart (Excel/Notion) |
| **4. PoC-test** | Identieke taak draaien op alle short-list kandidaten | Testrapport + metrics |
| **5. Onderbouwing** | Schriftelijke argumentatie + keuze + risico's | Beslisdocument |

---

## 2. Categorieën van tools (long-list)

| Categorie | Voorbeelden | Relevant voor node |
|---|---|---|
| **Agent-frameworks** | LangGraph, CrewAI, Microsoft Agent Framework, AutoGen, Semantic Kernel | Alle |
| **LLMs** | GPT-4o, GPT-4.1, Claude Sonnet 4.5, Gemini 2.5 Pro, Llama 3.3 | 1, 3, 5, 6, 8, 9 |
| **Vector databases** | Qdrant, Pinecone, Weaviate, Chroma, pgvector | 1, 3, 5, 8, 9 |
| **Maps/Routing APIs** | Google Maps, Mapbox, OpenStreetMap (OSRM) | 2, 4, 7 |
| **Supermarktdata** | `bartmachielsen/supermarktconnector`, scraping-alternatieven | 5 |
| **Validatie/Eval** | RAGAS, DeepEval, LangSmith, AgentEval, Promptfoo | Cross-cutting |
| **Observability** | LangSmith, Langfuse, OpenTelemetry, Phoenix (Arize) | Cross-cutting |

---

## 3. Vergelijkingscriteria (rubric)

Voor elke categorie hanteren we **dezelfde 7 dimensies**, met een schaal **1–5** en een **gewicht**. De gewichten passen we per categorie aan.

| # | Criterium | Wat meten we? | Hoe meten? |
|---|---|---|---|
| C1 | **Functionele dekking** | Ondersteunt de tool de 7 capabilities (workflows, orchestration, state, RAG, loops, tool calling, validatie)? | Feature-matrix uit documentatie |
| C2 | **Performance** | Latency, throughput, token-verbruik per taak | PoC-meting (p50/p95) |
| C3 | **Betrouwbaarheid** | Foutmarge, hallucinatierate, herstel na crash | Testset met 50 cases |
| C4 | **Developer Experience** | Setup-tijd, documentatie, error messages | Subjectieve score + tijdmeting |
| C5 | **Kosten** | Licentie + API-kosten + hosting per 1000 runs | Pricing-pages + meting |
| C6 | **Community & toekomst** | GitHub-stars, commits/maand, issues-response, vendor lock-in | GitHub-metrics, release notes |
| C7 | **Compliance / Privacy** | AVG, data-residency, op-prem optie | Vendor docs + DPA |

**Eindscore** = $$\sum_{i=1}^{7} w_i \cdot s_i$$ met $$\sum w_i = 1$$.

### Voorbeeld gewichtenset voor *Agent-frameworks*

| Criterium | Gewicht | Reden |
|---|---|---|
| C1 Functionele dekking | 0,25 | Hoofdcriterium |
| C2 Performance | 0,10 | Acceptabel als < 30s/run |
| C3 Betrouwbaarheid | 0,20 | Loops + guardrails kritisch |
| C4 DX | 0,15 | Studentenproject, leercurve telt |
| C5 Kosten | 0,10 | Open-source dus laag |
| C6 Community | 0,15 | Risico op end-of-life |
| C7 Compliance | 0,05 | PoC, geen productie |

---

## 4. Hoe vergelijken we precies? — De PoC-methode

### 4.1 Gouden regel: **één testtaak, alle kandidaten**

Definieer **één representatieve taak** die alle 7 capabilities raakt. Voorbeeld:

> *"Plan boodschappen voor pasta + salade + ontbijt voor 4 personen, budget €40, locatie 2511 CV Den Haag, voorkeur AH of Jumbo, max 15 minuten reistijd."*

Deze taak draaien we **identiek** op elke kandidaat (LangGraph vs CrewAI vs MAF) met **dezelfde LLM** (bv. GPT-4o) als controlevariabele.

### 4.2 Testset-opbouw

| Type case | Aantal | Doel |
|---|---|---|
| **Happy path** | 10 | Baseline werking |
| **Edge cases** | 15 | Geen winkels, budget te laag, allergie-conflict |
| **Adversarial** | 10 | Prompt injection, conflicterende eisen |
| **Loop-triggers** | 10 | Forceer Node 4→2, 5→3, 6→4 |
| **HITL** | 5 | Gebruiker wijst advies af |
| **Totaal** | **50** | |

### 4.3 Meetbare metrics per run

| Metric | Definitie | Tool |
|---|---|---|
| **Success rate** | % cases met geldig advies binnen budget+dekking | Eigen scoring-script |
| **Faithfulness** | % beweringen in advies dekkend in input | RAGAS |
| **Latency p50/p95** | Tijd van Node 1 → Node 8 | LangSmith / OpenTelemetry |
| **Token-cost** | $/run gemiddeld | LLM-API logs |
| **Loop-correctness** | % cases die juiste loop triggeren | Eigen unit tests |
| **Recovery rate** | % cases herstel na geforceerde crash | Chaos-test script |
| **Hallucination rate** | % runs met fictieve winkel/prijs | LLM-as-judge (Claude) |

### 4.4 Controle van confounders

- Zelfde **LLM-versie**, **seed/temperature** (bv. 0,2), **systeemprompt**.
- Zelfde **vector DB** content tijdens framework-vergelijking.
- Zelfde **API-keys** voor Google Maps en `supermarktconnector`.
- Wissel **één variabele per testronde** (factorial design).

---

## 5. Specifieke vergelijkingsmethode per categorie

### 5.1 Agent-frameworks
- **Methode:** identieke 9-node implementatie bouwen in elk framework.
- **Hard kill-criteria:** geen cyclische workflows → uitgesloten; geen HITL → uitgesloten.
- **Onderbouwing:** code-LOC vergelijking + DX-logboek.

### 5.2 LLMs (vooral Node 6 & 8)
- **Methode:** **A/B/C-test** met identieke prompt op GPT-4o vs Claude Sonnet 4.5 vs Gemini 2.5 Pro.
- **Beoordeling:**
  1. **Automatisch:** RAGAS faithfulness + answer relevance.
  2. **LLM-as-judge:** een vierde model (bv. GPT-4.1) scoort blind op 1–5 voor toon, beknoptheid, correctheid.
  3. **Menselijk:** 2 reviewers scoren 20 outputs (Cohen's kappa berekenen voor overeenkomst).

$$\kappa = \frac{p_o - p_e}{1 - p_e}$$

- **Drempel:** model wint als score ≥ 10% hoger én κ ≥ 0,6.

### 5.3 Vector databases
- **Methode:** zelfde 1000 product-embeddings indexeren in elke DB; 100 queries draaien.
- **Metrics:** Recall@5, query-latency, indexeerings-tijd, hosting-kosten.

### 5.4 Maps APIs
- **Methode:** zelfde 20 adressen door elke API; vergelijk geocoding-nauwkeurigheid (afstand tot ground truth in meters) en kosten/1000 calls.

### 5.5 Supermarktdata
- **Methode:** trek 50 producten uit `supermarktconnector`, vergelijk met handmatige prijzen op site (op zelfde dag). Bereken **prijs-match-rate** en **data-freshness** (timestamp delta).

---

## 6. Schriftelijke onderbouwing — Format

Per onderzochte tool produceren we **één pagina** met deze structuur:

```markdown
# [Tool-naam] – Evaluatie

## 1. Context
Wat is het, sinds wanneer, door wie ontwikkeld.

## 2. Feature-matrix (C1)
Tabel: ondersteunt capability X? Ja/Nee + bronlink.

## 3. PoC-resultaten (C2–C3)
Tabel met metrics uit sectie 4.3.

## 4. DX-logboek (C4)
- Setup-tijd: X uur
- Aantal blockers: Y
- Quote uit ervaring

## 5. Kosten (C5)
Berekening per 1000 runs.

## 6. Community (C6)
GitHub stars/commits, laatste release, issue-response-tijd.

## 7. Risico's & Beperkingen
Vendor lock-in, breaking changes, EOL-risico.

## 8. Eindscore
Gewogen score + ranking.

## 9. Bronnen
[doc1] ... [doc-n] — primaire documentatie + benchmarks.
```

### Bronnen-discipline

- Elke bewering met `[docN]` referentie.
- Geen Wikipedia/Medium zonder cross-check tegen officiële docs.
- Bewaar **screenshots/PDFs** van pricing-pages (snapshots vergaan).
- Gebruik **datum van raadpleging** voor elke URL.

---

## 7. Testen — Praktische opzet

### 7.1 Repo-structuur

```
/boodschappen-agent-eval/
├── frameworks/
│   ├── langgraph/   (zelfde 9 nodes)
│   ├── crewai/
│   └── maf/
├── testcases/
│   ├── happy_path.jsonl
│   ├── edge_cases.jsonl
│   └── adversarial.jsonl
├── eval/
│   ├── ragas_runner.py
│   ├── llm_judge.py
│   └── metrics.py
├── results/
│   └── 2026-06-XX-run-001.csv
└── report/
    └── final-comparison.md
```

### 7.2 CI-pipeline

1. **GitHub Actions** triggert nightly run.
2. Elke framework krijgt zelfde testset.
3. Resultaten → CSV → dashboard (Streamlit of Langfuse).
4. **Regression-alert** als success-rate > 5% daalt vs vorige run.

### 7.3 Statistische validiteit

- Minimum **n=50** runs per kandidaat voor stabiele gemiddelden.
- Bereken **95%-betrouwbaarheidsinterval** op success rate:

$$CI = \hat{p} \pm 1{,}96 \cdot \sqrt{\frac{\hat{p}(1-\hat{p})}{n}}$$

- Verschil tussen kandidaten is **significant** als CI's niet overlappen of via **McNemar's test** (paired binary outcomes).

---

## 8. Risico's in de onderzoeksopzet

| Risico | Mitigatie |
|---|---|
| Testtaak te smal → niet generaliseerbaar | Diverse cases (50+), meerdere persona's |
| LLM-as-judge bias (zelfde vendor) | Cross-model judging + menselijke steekproef |
| API-rate limits verstoren timing | Caching layer + retries met backoff |
| Framework-versies wijzigen tussen runs | Pin versies in `requirements.txt` / lockfile |
| Kosten lopen op | Budgetcap per run + dry-run mode |

---

## 9. Deliverables (wat lever je op?)

1. **Long-list & short-list document** (sectie 2).
2. **Ingevulde scorekaart** per kandidaat (sectie 3).
3. **PoC-repository** met 9-node implementatie in 2–3 frameworks.
4. **Testrapport** met metrics + grafieken (per categorie).
5. **Beslisdocument** (max 5 p.) met definitieve keuze + onderbouwing.
6. **Reproduceerbaarheids-bijlage** (commit-hashes, modelversies, datums).

---

## 10. Tijdlijn-suggestie (4 weken)

| Week | Activiteit |
|---|---|
| 1 | Scoping + desk research + scorekaart opstellen |
| 2 | PoC bouwen in 2 frameworks (parallel, in tweetallen) |
| 3 | Testset draaien + metrics verzamelen |
| 4 | Onderbouwing schrijven + presentatie + beslisdocument |

---

## Samenvatting

> **Vergelijk niet op gevoel, maar op een vooraf vastgestelde rubric met meetbare PoC-resultaten.**
> Eén testtaak + 50 cases + 7 metrics + gewogen scorekaart = onderbouwde keuze die je schriftelijk kunt verdedigen.
